[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/geometry/fitting/fitting.ipynb)

# Fitting Points, Lines and Planes

Given a cloud of noisy measurements, which point, line or plane fits them best? The usual answers are three different recipes: the centroid for a point, the principal axis for a line, the direction of least spread for a plane, and yet another formula for the point closest to a bundle of lines. Here all four are one function.

A sample and an unknown are joined: a point joined with a plane is a number, its signed distance; joined with a line it is the plane through both, whose size is the distance; joined with a point it is the line through both. The join is linear in the unknown, so its squared size summed over the samples is a quadratic form on the unknown's type, and the best fit is that form's flattest direction, its eigenvector of smallest value. The join does not care which of its arguments is a point and which a line, so the same function fitted to a bundle of lines triangulates their point of closest approach.

In the matrix notation of least squares the four fits read as the mean of the samples, the leading singular vector of the centred samples, the trailing one, and the solution of $\sum_k (I - \mathbf d_k \mathbf d_k^{\mathsf T})\,\mathbf x = \sum_k (I - \mathbf d_k \mathbf d_k^{\mathsf T})\,\mathbf p_k$ for lines through $\mathbf p_k$ along $\mathbf d_k$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import Extensor, GAType, NumpyContext, stack
from numga.algebras import PGA3D
from examples.util.animation import save_figure
from examples.geometry.fitting import render
from examples.geometry.fitting.core import bundle, cloud, end_planes, jitter, patch, patch_edges, segment

mv = NumpyContext(PGA3D).multivector
Point = PGA3D.gatype.antivector()
# A line through points is their join, intrinsic like them.
Line = PGA3D.gatype.antibivector()
Plane = PGA3D.gatype.vector()

## 1. One fit for every primitive

The unknown's type, `Point`, `Line` or `Plane`, is passed in as the identity map on it, so `samples & Unknown` is each sample's join with any element of that type, and its squared norm summed over the samples the form to minimise. The form's eigenvectors are normalised by the type's own metric, which fixes a plane's normal, a line's direction or a point's weight to unit size and leaves the position free for the fit to choose.

In [ ]:
def fit(Unknown: GAType, samples: Extensor) -> Extensor:
    """Fit a PGA primitive to a batch of points or lines by squared join distance.

    Unknown is the type solved for: points, lines or planes.
    """
    samples = samples.normalized()
    # Joining each sample with the unknown measures its incidence error:
    # point & plane is a scalar, point & line a plane, point & point a line.
    residual = samples & Unknown
    misfit = residual.scalar_norm_squared().sum(axis=0)                  # [] Scalar <- (Unknown, Unknown)

    # The unknown's own metric, the scalar part of the primitive's reverse product, fixes its
    # geometric size, leaving its position free: plane normal, line direction, or point weight
    # has unit norm. For a line it leaves out the condition to be a line, the pseudoscalar part;
    # the fitted moment is the centroid's moment about the fitted direction, which meets that
    # condition. The metric is blind to the ideal components, so for each mode eigh chooses them
    # to minimize the misfit:
    values, modes = misfit.eigh()                                        # [modes] Scalar, [modes] Unknown
    return modes[values.argmin()]

In [ ]:
# The pose every scene is placed by: a shift, after turns in the yz and xy planes; the seed of every scene's
# samples, and the size of the noise that moves each sample point.
shift, tilt, spin = np.array([0.4, -0.3, 0.6]), 0.3, 0.5
pose = ((stack((mv.xw, mv.yw, mv.zw)) * shift).sum(axis=0)).exp() * (mv.yz * tilt).exp() * (mv.xy * spin).exp()
seed, noise = 0, 0.05
origin = mv.w.dual()

In [ ]:
def distance(point: Point, primitive):
    """The distance of points of unit weight from a unit point, line or plane: the norm of their join."""
    return (point.normalized() & primitive.normalized()).norm()

## 2. A point to a cloud

Joined with a point, each sample gives the line through both, whose norm is their distance; the fit is the centroid. The noisy cloud, grey, with the true centre, a blue cross, and the fitted one, red.

In [ ]:
n_cloud, cloud_spread = 200, 0.5
rng = np.random.default_rng(seed)
points = jitter(pose >> cloud(n_cloud, cloud_spread, rng), noise, rng)         # [samples] Point
centroid = fit(Point, points)                                                  # [] Point
save_figure(render.draw_point_fit(points, pose >> origin, centroid), "fitting_point_to_points");

## 3. A line to a segment

Joined with a line, each sample gives the plane through both, whose norm is the sample's distance from the line. The samples, grey, with the true line, dashed blue, and the fitted one, red, each clipped by two end planes.

In [ ]:
n_segment, half_length, drawn_half_length = 120, 2.0, 2.5
rng = np.random.default_rng(seed)
points = jitter(pose >> segment(n_segment, half_length), noise, rng)           # [samples] Point
line = fit(Line, points)                                                       # [] Line
true_line = pose >> (origin & mv.direction([0, 1, 0]))                         # [] Line
ends = pose >> end_planes(drawn_half_length)                                   # [2] Plane
save_figure(render.draw_line_fit(points, true_line ^ ends, line ^ ends), "fitting_line_to_points");

## 4. A plane to a patch

Joined with a plane, each sample gives a number, its signed distance. The samples, grey, with the true plane, dashed blue, and the fitted one, red, each cut into a quad by four edge lines.

In [ ]:
n_patch, half_width = 200, 2.0
rng = np.random.default_rng(seed)
points = jitter(pose >> patch(n_patch, half_width, rng), noise, rng)           # [samples] Point
plane = fit(Plane, points)                                                     # [] Plane
true_plane = pose >> mv.z                                                      # [] Plane
edges = pose >> patch_edges(half_width)                                        # [4] Line
save_figure(render.draw_plane_fit(points, true_plane ^ edges, plane ^ edges), "fitting_plane_to_points");

## 5. A point to a bundle of lines

The samples are lines now, and the same function fits a point to them: joined with a point, each line gives the plane through both, whose norm is the point's distance from the line. The bundle, grey, each line drawn through the fitted point, with the true point, a blue cross, and the fitted one, red.

In [ ]:
n_rays, ray_spread, ray_half_length = 30, 0.05, 2.0
rng = np.random.default_rng(seed)
rays = pose >> bundle(n_rays, ray_spread, rng)                                 # [rays] Line
intersection = fit(Point, rays).normalized()                                   # [] Point
# Each line's point at infinity is its direction; its length is the norm of its complement.
directions = rays ^ mv.w                                                       # [rays] Point
unit_directions = directions / directions.dual().norm()
ray_ends = intersection + unit_directions * np.array([[-ray_half_length], [ray_half_length]])   # [2, rays] Point
save_figure(render.draw_bundle_fit(ray_ends, pose >> origin, intersection), "fitting_point_to_lines");

In [ ]:
# checks
# The samples limit each fit: the centre within 10 cm, the line's and plane's drawn ends within 5 cm of the
# truth, and the triangulated point within 5 cm.
assert distance(centroid, pose >> origin).to_array() < 0.1
assert distance(line ^ ends, true_line).to_array().max() < 0.05
assert distance(plane ^ edges, true_plane).to_array().max() < 0.05
assert distance(intersection, pose >> origin).to_array() < 0.05